# 실패 분석(Failure Analysis)

잘 작동하는 질문 몇 개만 보면 agent 시스템은 언제나 똑똑해 보인다. 하지만 실제 개선은 실패에서 시작된다. 이 노트북은 evaluation 결과 중 실패 케이스만 따로 뽑아, 어떤 단계(stage)에서 어떤 종류의 failure가 반복되는지 taxonomy와 trace를 통해 읽는 방법을 보여준다.

## 학습 목표
- failure taxonomy가 단순 버그 목록과 어떻게 다른지 이해한다.
- `retrieval_miss`, `bad_plan`, `ungrounded_synthesis` 같은 failure type을 자동 분류하는 흐름을 설명할 수 있다.
- severity(`critical`, `major`, `minor`)를 어떻게 해석해야 하는지 안다.
- trace를 열어 한 실패 케이스를 stage 단위로 역추적하는 습관을 익힌다.


## 개념 설명

failure taxonomy를 만드는 이유는 실패를 단순히 "틀렸다"로 끝내지 않기 위해서다. 실무에서 중요한 것은 같은 실패가 retrieval에서 반복되는지, planning에서 반복되는지, verifier가 너무 엄격한지처럼 개선 가능한 패턴으로 묶는 일이다. 즉 taxonomy는 버그 리포트보다 더 구조적이고, 개선 액션과 직접 연결된다.

**목적**
- 실패를 stage와 severity로 구조화해 읽는 관점을 만든다.

**핵심 로직**
- failure type은 stage와 연결된다.
- severity는 사용자 영향과 신뢰 리스크를 반영한다.
- trace는 한 failure가 실제로 어디서 발생했는지 보여준다.

**결과 해석 가이드**
- 이 notebook의 목표는 "실패를 줄이는 법"이지 "실패를 숨기는 법"이 아니다.

**💡 면접 포인트**
- "Failure taxonomy는 개별 버그를 묶어 반복 패턴과 개선 우선순위를 만드는 운영 도구"라고 설명할 수 있다.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import RuntimeConfig

print(sys.executable)
print(RuntimeConfig.auto_detect())

## Agent 시스템의 failure mode

먼저 evaluation 결과를 가져온다. 이미 저장된 `eval_results.json`이 있으면 재사용하고, 없으면 다시 평가를 돌린다. 이 설계가 중요한 이유는 failure 분석이 모델 실행 자체보다 재현 가능한 결과 기록에 기대기 때문이다.

**목적**
- 분석할 실패 후보 집합을 준비한다.

**핵심 로직**
- persisted 결과를 우선 읽어 재현성을 높인다.
- 결과가 없을 때만 evaluation suite를 다시 실행한다.

**결과 해석 가이드**
- `results.head()`는 단순 미리보기가 아니라, 어떤 컬럼이 failure 분류에 쓰이는지 확인하는 창이다.


## 구현

이 notebook에서 중요한 것은 raw result table을 바로 보는 것이 아니라, taxonomy와 analyzer를 통해 구조화된 질문으로 바꾸는 것이다. "이 실패는 어디서 생겼는가", "얼마나 치명적인가", "다음 개선 액션은 무엇인가"를 묻기 시작하면 분석 품질이 올라간다.

**결과 해석 가이드**
- 이후 셀의 표와 차트는 모두 같은 raw results에서 파생되므로, upstream 결과가 비어 있으면 downstream 분석도 의미가 없다.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from src.config import get_paths
from src.evaluator import attach_failure_improvements, extract_failure_cases, run_evaluation_suite
from src.failure_analyzer import analyze_failures, generate_failure_report, taxonomy_frame
from src.trace_debug import display_trace
from src.utils import read_json

paths = get_paths()
results_path = paths.eval_dir / 'eval_results.json'
if results_path.exists():
    results = pd.DataFrame(read_json(results_path))
else:
    results, _ = run_evaluation_suite(repeats=2, persist_outputs=True)

results.head(5)

## 실패 taxonomy

아래 taxonomy는 이 저장소가 현재 추적하는 실패 유형의 공식 사전이다. 각 failure type은 이름만 있는 것이 아니라, 설명, severity, stage, detection method, typical cause, mitigation까지 함께 가진다. 이 구조 덕분에 failure가 바로 개선 아이디어로 이어진다.

**목적**
- 어떤 실패를 어떤 기준으로 분류하는지 사전 수준에서 확인한다.

**핵심 로직**
- severity는 사용자 영향과 신뢰 리스크 기준이다.
- stage는 주된 발생 지점이다.
- mitigation은 다음 개선 액션이다.

**실제 소스 코드: FAILURE_TAXONOMY — src/failure_taxonomy.py**
```python
FAILURE_TAXONOMY: dict[str, FailureMetadata] = {
    "retrieval_miss": {
        "name": "Retrieval Miss",
        "description": "Relevant evidence never entered the retrieved context window.",
        "severity": "major",
        "stage": "retrieve_docs",
        "detection_method": "retrieval_hit_rate is 0.0 for a sample that expected grounded evidence.",
        "typical_cause": "Chunking is too coarse, lexical overlap is weak, or retrieval scoring is poorly tuned.",
        "mitigation": "Tune chunking, add richer retrieval features, or expand the corpus coverage.",
    },
    "retrieval_noise": {
        "name": "Retrieval Noise",
        "description": "Some evidence was retrieved, but the retrieved set was diluted by low-value chunks.",
        "severity": "minor",
        "stage": "retrieve_docs",
        "detection_method": "retrieval_hit_rate is partial and answer quality remains low.",
        "typical_cause": "Top-k is too large, ranking features are weak, or the query is underspecified.",
        "mitigation": "Tighten ranking, reduce noisy chunks, or add reranking before synthesis.",
    },
    "query_misclassification": {
        "name": "Query Misclassification",
        "description": "The workflow selected the wrong question type for the query.",
        "severity": "major",
        "stage": "classify_query",
        "detection_method": "predicted_question_type differs from expected_question_type.",
        "typical_cause": "Rule-based intent detection missed the dominant query signal.",
        "mitigation": "Expand classifier rules or replace them with a learned classifier.",
    },
    "bad_plan": {
        "name": "Bad Plan",
        "description": "The chosen reasoning template was not sufficient for the task.",
        "severity": "major",
        "stage": "make_plan",
        "detection_method": "answer quality is low without a better upstream explanation.",
        "typical_cause": "The workflow used a shallow plan for a multi-step or comparison-heavy question.",
        "mitigation": "Add stronger query-type-specific decomposition templates.",
    },
    "missing_decomposition": {
        "name": "Missing Decomposition",
        "description": "A multi-step question was handled without enough explicit reasoning steps.",
        "severity": "major",
        "stage": "make_plan",
        "detection_method": "average_steps is low for a complex question and the answer remains weak.",
        "typical_cause": "Planner templates are too short for multi-hop or comparison questions.",
        "mitigation": "Force decomposition for complex queries and expose intermediate sub-goals.",
    },
    "tool_execution_error": {
        "name": "Tool Execution Error",
        "description": "A tool call failed or produced unusable output during the run.",
        "severity": "critical",
        "stage": "run_tools",
        "detection_method": "Trace or error logs contain tool execution failures.",
        "typical_cause": "Malformed arguments, parsing issues, or unsupported tool input.",
        "mitigation": "Validate tool requests before execution and add error-aware fallbacks.",
    },
    "synthesis_quality_gap": {
        "name": "Synthesis Quality Gap",
        "description": "The workflow retrieved enough evidence but failed to form a strong answer.",
        "severity": "major",
        "stage": "synthesize_answer",
        "detection_method": "answer_correctness is low despite high retrieval hit rate.",
        "typical_cause": "Sentence selection or answer templating failed to capture the important evidence.",
        "mitigation": "Tighten answer templates or add a post-synthesis rewrite step.",
    },
    "incomplete_synthesis": {
        "name": "Incomplete Synthesis",
        "description": "The answer captured part of the evidence but omitted key details.",
        "severity": "minor",
        "stage": "synthesize_answer",
        "detection_method": "answer quality is middling and evidence coverage is only partially reflected in the answer.",
        "typical_cause": "The synthesis prompt or ranking logic prioritized only the first relevant fact.",
        "mitigation": "Increase synthesis coverage requirements and summarize multiple evidence spans explicitly.",
    },
    "ungrounded_synthesis": {
        "name": "Ungrounded Synthesis",
        "description": "The workflow answered confidently without enough supporting evidence.",
        "severity": "critical",
        "stage": "verify_grounding",
        "detection_method": "grounding_pass is false while the workflow still returned an answer.",
        "typical_cause": "Verification thresholds are too permissive or unsupported claims slipped through synthesis.",
        "mitigation": "Tighten verifier thresholds and block unsupported claims before finalization.",
    },
    "citation_mismatch": {
        "name": "Citation Mismatch",
        "description": "The cited or referenced sources do not line up with the expected evidence sources.",
        "severity": "major",
        "stage": "verify_grounding",
        "detection_method": "Provided citations do not overlap with expected_sources.",
        "typical_cause": "Source attribution drifted away from the evidence used to answer.",
        "mitigation": "Attach citations directly from retrieved chunks and validate them before finalization.",
    },
    "insufficient_evidence_not_detected": {
        "name": "Missed Abstention",
        "description": "The workflow should have abstained but answered anyway.",
        "severity": "critical",
        "stage": "fallback_or_finalize",
        "detection_method": "expected_status is abstained but predicted_status is answered.",
        "typical_cause": "Fallback thresholds are too lenient for insufficient-evidence questions.",
        "mitigation": "Raise verifier and fallback thresholds so unsupported answers abstain earlier.",
    },
    "over_abstention": {
        "name": "Over Abstention",
        "description": "The workflow abstained despite having enough evidence to answer.",
        "severity": "major",
        "stage": "fallback_or_finalize",
        "detection_method": "expected_status is answered but predicted_status is abstained.",
        "typical_cause": "Fallback thresholds or verifier criteria are too strict.",
        "mitigation": "Relax abstention thresholds when retrieval and grounding signals are sufficient.",
    },
}
```

**코드 읽기 포인트**
- `retrieval_miss`: gold source가 아예 top-k에 안 들어온 경우다.
- `retrieval_noise`: 관련 청크는 있었지만 잡음이 너무 많아 synthesis가 흔들린 경우다.
- `query_misclassification`: 잘못된 query type으로 이후 전체 흐름이 빗나간 경우다.
- `bad_plan`/`missing_decomposition`: reasoning skeleton이 너무 얕은 경우다.
- `tool_execution_error`: 도구 단계 실패로 critical severity를 갖는다.
- `ungrounded_synthesis`/`citation_mismatch`: 답은 했지만 근거 관리가 틀어진 경우라 신뢰 리스크가 크다.
- `insufficient_evidence_not_detected`: abstain해야 하는데 답한 경우로 가장 위험한 축에 가깝다.
- `over_abstention`: 반대로 답할 수 있는데 너무 보수적으로 거절한 경우다.

**결과 해석 가이드**
- 색이 더 진한 `critical`은 사용자 신뢰를 직접 깎을 수 있으므로 먼저 본다.
- `minor`는 곧바로 위험하진 않아도 누적되면 품질 체감이 나빠질 수 있다.

**💡 면접 포인트**
- "Severity는 모델 품질이 아니라 제품 리스크 기준으로 정한다"고 설명하면 실무 감각을 보여줄 수 있다.


In [ ]:
taxonomy = taxonomy_frame().sort_values(['severity', 'stage', 'failure_type']).reset_index(drop=True)
severity_colors = {'critical': '#FEE2E2', 'major': '#FEF3C7', 'minor': '#DBEAFE'}

def color_row(row):
    color = severity_colors.get(row['severity'], '#FFFFFF')
    return [f'background-color: {color}' for _ in row]

taxonomy.style.apply(color_row, axis=1)

## 실패 케이스 추출(Extract failure cases)

이제 raw evaluation record에서 실제 failure를 자동 분류한다. 중요한 점은 이 분류가 한 건당 하나의 태그만 붙이는 것이 아니라, 필요하면 여러 failure type을 동시에 붙일 수 있다는 것이다. 예를 들어 retrieval miss와 bad_plan이 함께 일어날 수도 있다.

**목적**
- raw result를 개선 가능한 failure case table로 바꾼다.

**핵심 로직**
- `classify_failure()`는 metric과 상태를 보고 failure type list를 만든다.
- `extract_failure_cases()`는 `failure_type != none`인 행만 고른다.
- `attach_failure_improvements()`는 failure type을 mitigation 문장에 매핑한다.

**실제 소스 코드: classify_failure() — src/failure_analyzer.py**
```python
def classify_failure(record: dict[str, Any]) -> list[str]:
    failures: list[str] = []

    expected_status = str(record.get("expected_status", "answered"))
    predicted_status = str(record.get("predicted_status", "answered"))
    retrieval = float(record.get("retrieval_hit_rate", 0.0) or 0.0)
    answer_correctness = float(record.get("answer_correctness", 0.0) or 0.0)
    grounding_pass = bool(record.get("grounding_pass", False))
    predicted_question_type = record.get("predicted_question_type")
    expected_question_type = record.get("expected_question_type")
    average_steps = float(record.get("average_steps", 0.0) or 0.0)
    citations = {str(item) for item in record.get("citations", [])}
    expected_sources = {str(item) for item in record.get("expected_sources", [])}

    if expected_status == "abstained" and predicted_status != "abstained":
        _append_unique(failures, "insufficient_evidence_not_detected")
    if expected_status == "answered" and predicted_status == "abstained":
        _append_unique(failures, "over_abstention")
    if retrieval == 0.0:
        _append_unique(failures, "retrieval_miss")
    if predicted_question_type and expected_question_type and predicted_question_type != expected_question_type:
        _append_unique(failures, "query_misclassification")
    if _contains_tool_error(record):
        _append_unique(failures, "tool_execution_error")
    if record.get("system") == "agent_workflow" and not grounding_pass and predicted_status == "answered":
        _append_unique(failures, "ungrounded_synthesis")
    if citations and expected_sources and citations.isdisjoint(expected_sources):
        _append_unique(failures, "citation_mismatch")

    if answer_correctness < 0.45:
        if retrieval >= 1.0 and predicted_status == "answered":
            _append_unique(failures, "synthesis_quality_gap")
            if 0.2 <= answer_correctness < 0.45:
                _append_unique(failures, "incomplete_synthesis")
        elif predicted_status == "answered":
            _append_unique(failures, "bad_plan")

    if 0.0 < retrieval < 1.0 and answer_correctness < 0.45 and predicted_status == "answered":
        _append_unique(failures, "retrieval_noise")

    if (
        expected_question_type in {"comparison", "multi_hop", "summary"}
        and predicted_status == "answered"
        and answer_correctness < 0.6
        and average_steps <= 4.0
    ):
        _append_unique(failures, "missing_decomposition")

    return failures
```

**실제 소스 코드: extract_failure_cases() — src/evaluator.py**
```python
def extract_failure_cases(results: pd.DataFrame) -> pd.DataFrame:
    return results[results["failure_type"] != "none"].copy()
```

**실제 소스 코드: attach_failure_improvements() — src/evaluator.py**
```python
def attach_failure_improvements(failures: pd.DataFrame) -> pd.DataFrame:
    if failures.empty:
        failures["improvement_idea"] = []
        return failures
    failures = failures.copy()
    failures["improvement_idea"] = failures["failure_type"].map(failure_mitigation).fillna("Inspect trace manually.")
    return failures
```

**코드 읽기 포인트**
- `expected_status == "abstained" and predicted_status != "abstained"`면 missed abstention이다.
- `retrieval == 0.0`이면 retrieval_miss를 붙인다.
- `answer_correctness < 0.45`이면서 retrieval은 높으면 synthesis_quality_gap 쪽을 의심한다.
- improvement idea를 즉시 붙여 주면 failure table 자체가 다음 작업 목록 역할을 한다.

**결과 해석 가이드**
- 같은 failure type이 반복되면 개별 trace보다 먼저 해당 stage의 로직을 점검하는 것이 효율적이다.
- `improvement_idea`가 비슷하게 반복되면 구조적인 병목이 있다는 뜻이다.


In [ ]:
failures = attach_failure_improvements(extract_failure_cases(results))
failures[['system', 'question_id', 'expected_question_type', 'failure_type', 'improvement_idea']].head(12)

## trace 점검(Trace inspection)

실패를 유형별로 묶었다면, 이제 대표 사례를 trace로 내려가 본다. 이 단계가 중요한 이유는 taxonomy가 "무슨 종류의 실패인가"를 알려 주고, trace는 "실제로 어떤 입력에서 어떤 출력이 나왔는가"를 보여주기 때문이다. 둘이 합쳐져야 actionable한 분석이 된다.

**목적**
- failure distribution을 집계하고 대표 케이스 trace를 직접 본다.

**핵심 로직**
- `analyze_failures()`는 failure/stage/severity 분포와 top improvement action을 만든다.
- `generate_failure_report()`는 markdown 보고서를 저장한다.
- 대표 failure의 trace JSON을 열어 node별 입출력을 다시 본다.

**실제 소스 코드: analyze_failures() — src/failure_analyzer.py**
```python
def analyze_failures(results_df: pd.DataFrame) -> dict[str, Any]:
    if results_df.empty:
        return {
            "total_failure_instances": 0,
            "failure_distribution": {},
            "stage_distribution": {},
            "severity_distribution": {},
            "top_improvement_actions": [],
        }

    failure_records = _records_with_failures(results_df)
    failure_distribution = Counter(failure for failure, _ in failure_records)
    stage_distribution = Counter(metadata["stage"] for _, metadata in failure_records)
    severity_distribution = Counter(metadata["severity"] for _, metadata in failure_records)
    mitigation_distribution = Counter(failure_mitigation(failure) for failure, _ in failure_records)

    top_improvement_actions = [
        {"mitigation": mitigation, "count": count}
        for mitigation, count in mitigation_distribution.most_common(5)
    ]

    return {
        "total_failure_instances": sum(failure_distribution.values()),
        "failure_distribution": dict(sorted(failure_distribution.items())),
        "stage_distribution": dict(sorted(stage_distribution.items())),
        "severity_distribution": dict(sorted(severity_distribution.items())),
        "top_improvement_actions": top_improvement_actions,
    }
```

**코드 읽기 포인트**
- 단순 count보다 stage distribution이 중요하다. retrieval에 몰리면 retriever를, fallback에 몰리면 threshold를 손봐야 한다.
- trace를 볼 때는 "처음 잘못된 node"를 찾는 것이 핵심이다. downstream 실패는 upstream 오류의 결과일 수 있다.

**결과 해석 가이드**
- `trace_steps`가 짧다고 항상 좋은 것은 아니다. 중간에 조기 종료된 실패일 수도 있다.
- `failure_report.md`는 notebook 바깥에서도 같은 분석을 재사용하게 해 준다.

**💡 면접 포인트**
- "Taxonomy는 aggregate view, trace는 case study view다. 둘을 함께 봐야 개선 우선순위를 정할 수 있다"고 말할 수 있다.


In [ ]:
from pathlib import Path

analysis = analyze_failures(failures)
report_path = paths.reports_dir / 'failure_report.md'
generate_failure_report(analysis, report_path)

agent_failures = failures[failures['system'] == 'agent_workflow'].head(3)
trace_previews = []
for _, row in agent_failures.iterrows():
    trace_path = paths.traces_dir / f"{row['question_id']}_run{int(row['run_id'])}.json"
    trace = read_json(trace_path)['trace'] if trace_path.exists() else []
    trace_previews.append(
        {
            'question_id': row['question_id'],
            'failure_type': row['failure_type'],
            'trace_path': str(trace_path),
            'trace_steps': len(trace),
        }
    )

trace_preview_frame = pd.DataFrame(trace_previews)
display(trace_preview_frame)
for preview in trace_previews:
    print(f"Trace for {preview['question_id']} ({preview['failure_type']})")
    display_trace(read_json(Path(preview['trace_path']))['trace'])

## 실험

stage별 막대 그래프와 severity pie chart는 failure landscape를 빠르게 읽게 해 준다. 이 시각화의 목적은 "가장 많이 발생한 것"과 "가장 위험한 것"을 동시에 보는 것이다. 빈도와 심각도가 항상 같지 않기 때문이다.

**목적**
- 실패 분포를 시각적으로 읽는다.

**결과 해석 가이드**
- stage bar chart에서 특정 stage가 튀면 그 stage가 현재 병목이다.
- severity pie에서 critical 비율이 작더라도, 사용자 신뢰를 직접 해치는 failure라면 우선순위는 여전히 높을 수 있다.


In [ ]:
stage_distribution = pd.Series(analysis['stage_distribution']).sort_values(ascending=False)
severity_distribution = pd.Series(analysis['severity_distribution']).sort_values(ascending=False)
improvement_actions = pd.DataFrame(analysis['top_improvement_actions'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
stage_distribution.plot(kind='bar', color='#4C78A8', ax=axes[0], title='Failures by Stage')
axes[0].set_xlabel('stage')
axes[0].set_ylabel('count')
severity_distribution.plot(kind='pie', autopct='%1.0f%%', ax=axes[1], title='Failure Severity Mix')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

display(improvement_actions)

## 결과 해석

마지막 요약 표는 failure 분석을 실제 액션으로 바꾸기 위한 최소 대시보드다. total failure 수, unique failure type 수, top mitigation을 함께 보면 "지금 무엇부터 고칠 것인가"가 보이기 시작한다.

**목적**
- failure analysis를 우선순위와 연결한다.

**결과 해석 가이드**
- `top_improvement_action`은 단순 문장이 아니라, 다음 iteration의 가장 유력한 개선 후보로 읽으면 된다.
- failure 수가 줄어도 critical이 남아 있으면 아직 안전한 시스템이라고 말하기 어렵다.


In [ ]:
pd.Series({
    'total_failures': int(analysis['total_failure_instances']),
    'unique_failure_types': len(analysis['failure_distribution']),
    'report_path': str(report_path),
    'top_improvement_action': improvement_actions.iloc[0]['mitigation'] if not improvement_actions.empty else 'None',
})

## 핵심 정리

이 노트북을 통해 failure analysis는 "틀린 답 몇 개 모아보기"가 아니라, 실패를 stage·severity·mitigation 관점에서 구조화하는 작업이라는 점을 확인했다. `FAILURE_TAXONOMY`는 현재 시스템이 추적하는 실패 사전이고, `classify_failure()`와 `analyze_failures()`는 raw evaluation result를 개선 가능한 작업 목록으로 바꿔 준다. 그 뒤 trace를 열어 대표 사례를 보면, aggregate view와 case study view가 연결된다.

실무에서 중요한 것은 failure를 숨기지 않고, 반복 패턴을 잡아 다음 iteration의 우선순위로 바꾸는 것이다. retrieval miss가 많으면 retriever와 chunking을, missed abstention이 많으면 verifier와 fallback threshold를 우선 손보는 식이다.

**💡 면접 포인트**
- "Failure taxonomy는 버그 목록이 아니라, stage·severity·mitigation이 연결된 운영 프레임워크다."
- "Aggregate chart로 병목을 찾고, 대표 trace로 원인을 확인하는 두 단계 분석이 중요하다."
- "Critical failure는 빈도가 낮아도 신뢰 리스크가 크므로 우선적으로 해결해야 한다."
